# TrustOCT — Notebook 4: Final Analysis & Paper Results
### Combines all 6 model results into ablation tables, comparison plots, and paper figures
---
Run this notebook **after all 3 training notebooks have completed** and synced their results to the shared Google Drive folder.

In [ ]:
import os

if not os.path.exists('src'):
    !git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git
    %cd Trusthworthy_OCTAI
else:
    print('Repository already cloned.')

try:
    import albumentations
    import ptflops
except ImportError:
    !pip install -r requirements.txt

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, classification_report, matthews_corrcoef
from scipy.stats import wilcoxon

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')

In [ ]:
# 1. Pull latest repo updates
!git pull

import os, shutil, torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from src.models.builder import build_model
from src.datasets.dataset import auto_detect_columns, patient_level_split, RetinalDataset
from src.preprocessing.standardizer import RetinalPipelineTransform
from src.models.edl_head import get_evidence_metrics
from src.evaluation.plots import plot_confusion_matrix, plot_reliability_diagram

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception:
    print('Running locally or Drive skipped.')

# 2. Scan Google Drive for saved model weights
weight_dirs = [
    '/content/drive/MyDrive/TrustOCT_weights',
    '/content/drive/MyDrive/TrustOCT_weights (1)',
    '/content/drive/MyDrive/TrustOCT_weights (2)',
    '/content/drive/MyDrive/TrustOCT_Results/outputs'
]

WEIGHT_FILES = {
    'resnet50.pth': 'outputs/resnet50/weights_best.pth',
    'resnet50_weights_best.pth': 'outputs/resnet50/weights_best.pth',
    'msf_resnet50.pth': 'outputs/msf_resnet50/weights_best.pth',
    'msf_resnet50_weights_best.pth': 'outputs/msf_resnet50/weights_best.pth',
    'msf_cbam_resnet50.pth': 'outputs/msf_cbam_resnet50/weights_best.pth',
    'msf_cbam_resnet50_weights_best.pth': 'outputs/msf_cbam_resnet50/weights_best.pth',
    'msf_cbam_mixstyle_resnet50.pth': 'outputs/msf_cbam_mixstyle_resnet50/weights_best.pth',
    'msf_cbam_mixstyle_resnet50_weights_best.pth': 'outputs/msf_cbam_mixstyle_resnet50/weights_best.pth',
    'msf_cbam_edl_resnet50.pth': 'outputs/msf_cbam_edl_resnet50/weights_best.pth',
    'msf_cbam_edl_resnet50_weights_best.pth': 'outputs/msf_cbam_edl_resnet50/weights_best.pth',
    'trustoct.pth': 'outputs/trustoct/weights_best.pth',
    'trustoct_weights_best.pth': 'outputs/trustoct/weights_best.pth'
}

print('Searching Google Drive for model weights...')
for wdir in weight_dirs:
    if os.path.exists(wdir):
        for fname in os.listdir(wdir):
            if fname in WEIGHT_FILES:
                src_path = os.path.join(wdir, fname)
                dst_path = WEIGHT_FILES[fname]
                os.makedirs(os.path.dirname(dst_path), exist_ok=True)
                shutil.copy(src_path, dst_path)
                print(f'  Loaded weights: {fname} -> {dst_path}')

# Create dataset CSV mapping if not present
csv_path = 'kermany_dataset_mapping.csv'
if not os.path.exists(csv_path):
    root_oct = '/content/Kermany/OCT2017 '
    if not os.path.exists(root_oct): root_oct = '/content/Kermany'
    if not os.path.exists(root_oct): root_oct = '/content/OCT2017'
    
    if os.path.exists(root_oct):
        records = []
        class_to_idx = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3}
        for root, dirs, files_list in os.walk(root_oct):
            for f in files_list:
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    parent_dir = os.path.basename(root)
                    lbl = class_to_idx.get(parent_dir.lower(), -1)
                    if lbl != -1:
                        base = os.path.splitext(f)[0]
                        parts = base.split('-')
                        pt_id = '-'.join(parts[:2]) if len(parts) >= 2 else base
                        records.append({'image_path': os.path.join(root, f), 'label': lbl, 'patient_id': pt_id})
        df_new = pd.DataFrame(records)
        df_new = df_new[df_new['label'] != -1]
        df_new.to_csv(csv_path, index=False)
        print(f'Created mapping CSV with {len(df_new)} images!')

# 3. Instant prediction generator for any missing CSV files (takes 30 seconds total!)
if os.path.exists(csv_path):
    df = auto_detect_columns(pd.read_csv(csv_path))
    is_colab = os.path.exists('/content')
    local_kermany = '/content/Kermany'
    local_oct2017 = '/content/OCT2017'
    if is_colab and (os.path.exists(local_kermany) or os.path.exists(local_oct2017)):
        def force_local_path(path_str):
            p = path_str.replace('\\', '/').replace('//', '/')
            parts = p.split('/')
            for folder in ['train', 'val', 'test']:
                if folder in parts:
                    idx = parts.index(folder)
                    subpath = '/'.join(parts[idx:])
                    for candidate in [
                        os.path.join(local_kermany, subpath),
                        os.path.join(local_kermany, 'OCT2017', subpath),
                        os.path.join(local_kermany, 'OCT2017 ', subpath),
                        os.path.join(local_oct2017, subpath)
                    ]:
                        if os.path.exists(candidate): return candidate
            return path_str
        df['image_path'] = df['image_path'].apply(force_local_path)
    _, _, test_df = patient_level_split(df)
    
    transform_val = RetinalPipelineTransform(is_training=False)
    test_dataset = RetinalDataset(test_df, transform=transform_val, apply_bilateral=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
    
    models_config = [
        ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'resnet50'),
        ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'msf_resnet50'),
        ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'msf_cbam_resnet50'),
        ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'msf_cbam_mixstyle_resnet50'),
        ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'msf_cbam_edl_resnet50'),
        ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'trustoct')
    ]
    
    for config_item, exp_id in models_config:
        weights_path = f'outputs/{exp_id}/weights_best.pth'
        pred_csv = f'outputs/{exp_id}/predictions.csv'
        if os.path.exists(weights_path) and not os.path.exists(pred_csv):
            print(f'Generating predictions.csv for {exp_id} from saved weights...')
            model_inst = build_model(config_item).to(device)
            model_inst.load_state_dict(torch.load(weights_path, map_location=device))
            model_inst.eval()
            is_evidential = (config_item['model']['head'] == 'evidential')
            all_true, all_pred, all_probs, all_uncertainties = [], [], [], []
            with torch.no_grad():
                for images, targets in test_loader:
                    images = images.to(device)
                    outputs = model_inst(images)
                    if is_evidential:
                        probs, _ = get_evidence_metrics(outputs)
                        evidence = torch.relu(outputs)
                        alpha = evidence + 1.0
                        uncertainty = 4.0 / torch.sum(alpha, dim=1)
                    else:
                        probs = torch.softmax(outputs, dim=1)
                        ent = -torch.sum(probs * torch.log(probs + 1e-8), dim=1)
                        uncertainty = ent / np.log(4.0)
                    preds_idx = torch.argmax(probs, dim=1)
                    all_probs.extend(probs.cpu().numpy())
                    all_pred.extend(preds_idx.cpu().numpy())
                    all_true.extend(targets.numpy())
                    all_uncertainties.extend(uncertainty.cpu().numpy())
            pred_records = []
            for i in range(len(all_true)):
                row = test_df.iloc[i]
                pred_records.append({
                    'image_path': row.get('image_path', ''),
                    'patient_id': row.get('patient_id', ''),
                    'true_label': CLASSES[all_true[i]],
                    'pred_label': CLASSES[all_pred[i]],
                    'prob_0': all_probs[i][0], 'prob_1': all_probs[i][1],
                    'prob_2': all_probs[i][2], 'prob_3': all_probs[i][3],
                    'uncertainty': all_uncertainties[i]
                })
            df_pred = pd.DataFrame(pred_records)
            os.makedirs(f'outputs/{exp_id}', exist_ok=True)
            df_pred.to_csv(pred_csv, index=False)
            plot_confusion_matrix(np.array(all_true), np.array(all_pred), CLASSES, f'outputs/{exp_id}/confusion_matrix.png')
            plot_reliability_diagram(np.max(np.array(all_probs), axis=1), (np.array(all_pred) == np.array(all_true)).astype(int), num_bins=10, save_path=f'outputs/{exp_id}/reliability_diagram.png')
            print(f'  ✅ Generated predictions.csv & plots for {exp_id}')

# 4. Final audit status check
expected_models = ['resnet50', 'msf_resnet50', 'msf_cbam_resnet50', 'msf_cbam_mixstyle_resnet50', 'msf_cbam_edl_resnet50', 'trustoct']
print('\n--- MODEL RESULTS AUDIT ---')
for m in expected_models:
    pred_status = 'CSV FOUND' if os.path.exists(f'outputs/{m}/predictions.csv') else 'CSV MISSING'
    w_status = 'WEIGHTS FOUND' if os.path.exists(f'outputs/{m}/weights_best.pth') else 'NO WEIGHTS'
    print(f'  {m:<30}: {pred_status} | {w_status}')


## TABLE 2: Classification Performance (with 95% Bootstrap CIs)

In [ ]:
from src.evaluation.classification import evaluate_classification_metrics
from src.evaluation.calibration import calculate_ece, calculate_brier_score
from src.evaluation.plots import plot_confusion_matrix, plot_reliability_diagram
from src.models.builder import build_model

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
class_to_idx = {c: idx for idx, c in enumerate(CLASSES)}

def compute_all_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds = np.array(preds)
    probs = np.array(probs)
    
    perf = evaluate_classification_metrics(labels, preds)
    confidences = np.max(probs, axis=1)
    accuracies = (preds == labels).astype(int)
    
    ece = calculate_ece(confidences, accuracies)
    brier = calculate_brier_score(probs, labels)
    
    from src.evaluation.classification import compute_multiclass_specificity
    specificity = compute_multiclass_specificity(labels, preds)
    
    present_classes = sorted(list(np.unique(labels)))
    if len(present_classes) > 1:
        class_map = {old_label: new_label for new_label, old_label in enumerate(present_classes)}
        mapped_labels = [class_map[lbl] for lbl in labels]
        probs_sliced = probs[:, present_classes]
        row_sums = probs_sliced.sum(axis=1, keepdims=True)
        probs_sliced = np.where(row_sums > 1e-5, probs_sliced / row_sums, np.ones_like(probs_sliced) / probs_sliced.shape[1])
        auc = roc_auc_score(mapped_labels, probs_sliced, multi_class='ovr', average='macro')
    else:
        auc = 0.5
        
    return {
        'Accuracy': perf['accuracy'], 'Precision': perf['precision'], 'Recall': perf['recall'],
        'Specificity': specificity, 'Macro F1': perf['f1_score'], 'Kappa': perf['cohens_kappa'],
        'ROC-AUC': auc, 'ECE': ece, 'Brier': brier
    }

def load_predictions_and_bootstrap(pred_path, n_bootstraps=200):
    df_pred = pd.read_csv(pred_path)
    labels = df_pred['true_label'].map(class_to_idx).values
    preds = df_pred['pred_label'].map(class_to_idx).values
    probs = df_pred[['prob_0', 'prob_1', 'prob_2', 'prob_3']].values
    
    base_scores = compute_all_metrics(labels, preds, probs)
    bootstrap_results = {k: [] for k in base_scores.keys()}
    n_samples_len = len(labels)
    np.random.seed(42)
    for _ in range(n_bootstraps):
        boot_idx = np.random.choice(n_samples_len, size=n_samples_len, replace=True)
        boot_labels = labels[boot_idx]
        boot_preds = preds[boot_idx]
        boot_probs = probs[boot_idx]
        
        boot_scores = compute_all_metrics(boot_labels, boot_preds, boot_probs)
        for k, v in boot_scores.items():
            bootstrap_results[k].append(v)
            
    report = {}
    for k, base_val in base_scores.items():
        boot_vals = sorted(bootstrap_results[k])
        ci_lower = boot_vals[int(0.025 * n_bootstraps)]
        ci_upper = boot_vals[int(0.975 * n_bootstraps)]
        report[k] = base_val
        report[f'{k}_CI'] = (ci_lower, ci_upper)
        
    return report, preds, labels, probs

models_to_evaluate = [
    ('outputs/resnet50/predictions.csv', 'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    ('outputs/trustoct/predictions.csv', 'TrustOCT (Proposed)')
]

os.makedirs('results/tables', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)

comparison_rows = []
best_preds, best_labels, best_probs = None, None, None

for path, display_name in models_to_evaluate:
    if os.path.exists(path):
        print(f'Auditing {display_name} with bootstrap CIs...')
        report, preds, labels, probs = load_predictions_and_bootstrap(path)
        row = {'Model': display_name}
        for metric in ['Accuracy', 'Precision', 'Recall', 'Specificity', 'Macro F1', 'Kappa', 'ROC-AUC', 'ECE', 'Brier']:
            val = report[metric]
            ci = report[f"{metric}_CI"]
            row[metric] = f"{val:.4f} ({ci[0]:.4f} - {ci[1]:.4f})"
        comparison_rows.append(row)
        if display_name == 'TrustOCT (Proposed)':
            best_preds = preds
            best_labels = labels
            best_probs = probs
    else:
        print(f'Skipping {display_name}: predictions file {path} not found.')
        
if len(comparison_rows) > 0:
    table_2_df = pd.DataFrame(comparison_rows)
    print('\n--- TABLE 2: CORE MODELS COMPARISON (WITH 95% BOOTSTRAP CIs) ---')
    display(table_2_df)
    table_2_df.to_csv('results/tables/table_2_metrics_ci.csv', index=False)

## TABLE 3: Ablation Study

In [ ]:
ablation_configs = [
    ('outputs/resnet50/predictions.csv', 'resnet50'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    ('outputs/trustoct/predictions.csv', 'trustoct')
]

ablation_rows = []
for path, config_name in ablation_configs:
    if os.path.exists(path):
        df_pred = pd.read_csv(path)
        labels = df_pred['true_label'].map(class_to_idx).values
        preds = df_pred['pred_label'].map(class_to_idx).values
        
        acc = accuracy_score(labels, preds)
        _, _, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
        mcc = matthews_corrcoef(labels, preds)
        
        ablation_rows.append({
            'Configuration': config_name,
            'Accuracy (%)': f"{acc*100:.2f}%",
            'Macro F1': f"{f1:.4f}",
            'MCC': f"{mcc:.4f}"
        })
    else:
        print(f'Skipping {config_name}: not found.')
        
if len(ablation_rows) > 0:
    ablation_df = pd.DataFrame(ablation_rows)
    print('--- TABLE 3: ABLATION STUDY ---')
    display(ablation_df)
    ablation_df.to_csv('results/tables/table_3_ablation_study.csv', index=False)

## Confusion Matrix (TrustOCT Proposed)

In [ ]:
cm_path = 'outputs/trustoct/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image.open(cm_path))
else:
    print('Confusion matrix not found.')

## TABLE 4: Explainability Faithfulness (Deletion & Insertion)

In [ ]:
from src.explainability.comparison import calculate_saliency_entropy, run_deletion_test, run_insertion_test
from src.explainability.layercam import LayerCAM
from src.preprocessing.standardizer import RetinalPipelineTransform
from src.datasets.dataset import auto_detect_columns, patient_level_split

# Load dataset for test records
csv_path = 'kermany_dataset_mapping.csv'
shared_csv = '/content/drive/MyDrive/TrustOCT_Results/kermany_dataset_mapping.csv'
if not os.path.exists(csv_path) and os.path.exists(shared_csv):
    import shutil
    shutil.copy(shared_csv, csv_path)

if os.path.exists(csv_path):
    df = auto_detect_columns(pd.read_csv(csv_path))
    
    is_colab = os.path.exists('/content')
    local_kermany = '/content/Kermany'
    local_oct2017 = '/content/OCT2017'
    if is_colab and (os.path.exists(local_kermany) or os.path.exists(local_oct2017)):
        def force_local_path(path_str):
            p = path_str.replace('\\', '/').replace('//', '/')
            parts = p.split('/')
            for folder in ['train', 'val', 'test']:
                if folder in parts:
                    idx = parts.index(folder)
                    subpath = '/'.join(parts[idx:])
                    for candidate in [
                        os.path.join(local_kermany, subpath),
                        os.path.join(local_kermany, 'OCT2017', subpath),
                        os.path.join(local_kermany, 'OCT2017 ', subpath),
                        os.path.join(local_oct2017, subpath),
                        os.path.join(local_oct2017, 'OCT2017', subpath)
                    ]:
                        if os.path.exists(candidate):
                            return candidate
            return path_str
        df['image_path'] = df['image_path'].apply(force_local_path)
    _, _, test_df = patient_level_split(df)

val_transform = RetinalPipelineTransform(is_training=False)
test_records = test_df.to_dict('records')[:20]
comparison_rows = []
models_to_audit = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'ResNet-50 Baseline'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/trustoct/weights_best.pth', 'TrustOCT (Proposed)')
]

for config_item, path, display_name in models_to_audit:
    if os.path.exists(path):
        print(f"Auditing explainability for {display_name}...")
        model_inst = build_model(config_item).to(device)
        is_ev = (config_item['model']['head'] == 'evidential')
        model_inst.load_state_dict(torch.load(path, map_location=device))
        model_inst.eval()
        
        target_layer = model_inst.backbone.layer4[-1]
        cam_generator = LayerCAM(model_inst, target_layer)
        
        del_scores, ins_scores, entropies = [], [], []
        for rec in test_records:
            img_p = rec['image_path']
            if not os.path.exists(img_p):
                continue
            try:
                img_p_pil = Image.open(img_p).convert('RGB')
                tensor_inp = val_transform(img_p_pil).unsqueeze(0).to(device)
                with torch.no_grad():
                    outs = model_inst(tensor_inp)
                    if is_ev:
                        from src.models.edl_head import get_evidence_metrics
                        probs_ev, _ = get_evidence_metrics(outs)
                        pred_cls = probs_ev.argmax(dim=1).item()
                    else:
                        pred_cls = torch.softmax(outs, dim=1).argmax(dim=1).item()
                        
                cam_heatmap = cam_generator.generate(tensor_inp, class_idx=pred_cls)
                _, aopc_del, _ = run_deletion_test(model_inst, tensor_inp, cam_heatmap, class_idx=pred_cls)
                _, aopc_ins, _ = run_insertion_test(model_inst, tensor_inp, cam_heatmap, class_idx=pred_cls)
                entropy = calculate_saliency_entropy(cam_heatmap)
                
                del_scores.append(aopc_del)
                ins_scores.append(aopc_ins)
                entropies.append(entropy)
            except Exception:
                continue
                
        cam_generator.release()
        comparison_rows.append({
            'Model': display_name,
            'Deletion AOPC': f"{np.mean(del_scores):.4f} +/- {np.std(del_scores):.4f}",
            'Insertion AOPC': f"{np.mean(ins_scores):.4f} +/- {np.std(ins_scores):.4f}",
            'Saliency Entropy': f"{np.mean(entropies):.4f} +/- {np.std(entropies):.4f}"
        })
    else:
        print(f'Skipping {display_name}: checkpoint not found.')
        
if len(comparison_rows) > 0:
    table_4_df = pd.DataFrame(comparison_rows)
    print('\n--- TABLE 4: EXPLAINABILITY FAITHFULNESS BENCHMARKS ---')
    display(table_4_df)
    table_4_df.to_csv('results/tables/table_4_explainability_averages.csv', index=False)

## Statistical Significance (Wilcoxon + McNemar)

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

print('Running Statistical Significance Tests...')
results_cache = {}

configs_for_significance = [
    ('outputs/resnet50/predictions.csv', 'ResNet-50 Baseline'),
    ('outputs/trustoct/predictions.csv', 'TrustOCT (Proposed)')
]

for path, display_name in configs_for_significance:
    if os.path.exists(path):
        df_pred = pd.read_csv(path)
        labels = df_pred['true_label'].map(class_to_idx).values
        preds = df_pred['pred_label'].map(class_to_idx).values
        results_cache[display_name] = {'preds': np.array(preds), 'labels': np.array(labels)}

def run_mcnemar(labels, preds_a, preds_b):
    correct_a = (preds_a == labels)
    correct_b = (preds_b == labels)
    n00 = np.sum(~correct_a & ~correct_b)
    n01 = np.sum(~correct_a & correct_b)
    n10 = np.sum(correct_a & ~correct_b)
    n11 = np.sum(correct_a & correct_b)
    table = [[n11, n10], [n01, n00]]
    res = mcnemar(table, exact=True)
    return res.statistic, res.pvalue

proposed_name = 'TrustOCT (Proposed)'
if proposed_name in results_cache and 'ResNet-50 Baseline' in results_cache:
    prop_data = results_cache[proposed_name]
    base_data = results_cache['ResNet-50 Baseline']
    stat, p_val = run_mcnemar(prop_data['labels'], prop_data['preds'], base_data['preds'])
    print(f"McNemar's test proposed vs baseline ResNet50 p-value: {p_val:.5f}")
    
    diff = (prop_data['preds'] == prop_data['labels']).astype(int) - (base_data['preds'] == base_data['labels']).astype(int)
    if not np.all(diff == 0):
        stat_w, p_val_w = wilcoxon(diff)
        print(f"Wilcoxon proposed vs baseline ResNet50 p-value: {p_val_w:.5f}")

## Training Curves & Publication Figures

In [ ]:
print('Generating publication figure curves...')
history_files = {
    'resnet50': 'outputs/resnet50/metrics.csv',
    'msf_resnet50': 'outputs/msf_resnet50/metrics.csv',
    'msf_cbam_resnet50': 'outputs/msf_cbam_resnet50/metrics.csv',
    'trustoct': 'outputs/trustoct/metrics.csv'
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_to_plot = [
    ('train_loss', 'Training Loss', axes[0, 0]),
    ('val_loss', 'Validation Loss', axes[0, 1]),
    ('train_acc', 'Training Accuracy', axes[1, 0]),
    ('val_acc', 'Validation Accuracy', axes[1, 1])
]
for metric_key, title, ax in metrics_to_plot:
    for model_name, path in history_files.items():
        if os.path.exists(path):
            df_hist = pd.read_csv(path)
            ax.plot(df_hist['epoch'], df_hist[metric_key], label=model_name, linewidth=2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend()
plt.tight_layout()
plt.savefig('results/figures/figure_2_training_curves.png', dpi=300)
plt.show()

## TABLE 6: Computational Complexity

In [ ]:
import time, os, torch
import pandas as pd
from src.models.builder import build_model

complexity_rows = []
dummy_input = torch.randn(1, 3, 224, 224).to(device)
models_for_complexity = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'ResNet-50 Baseline'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_resnet50/weights_best.pth', 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_resnet50/weights_best.pth', 'msf_cbam_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_mixstyle_resnet50/weights_best.pth', 'msf_cbam_mixstyle_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_edl_resnet50/weights_best.pth', 'msf_cbam_edl_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/trustoct/weights_best.pth', 'TrustOCT (Proposed)')
]

for config_item, path, display_name in models_for_complexity:
    model_inst = build_model(config_item).to(device)
    if os.path.exists(path):
        model_inst.load_state_dict(torch.load(path, map_location=device))
        model_size_mb = os.path.getsize(path) / (1024 * 1024)
        size_str = f"{model_size_mb:.2f} MB"
    else:
        # Approximate size from parameter count (4 bytes per float32 param)
        total_p = sum(p.numel() for p in model_inst.parameters())
        size_str = f"~{total_p * 4 / (1024*1024):.2f} MB"
        
    model_inst.eval()
    total_params = sum(p.numel() for p in model_inst.parameters())
    trainable_params = sum(p.numel() for p in model_inst.parameters() if p.requires_grad)
    
    with torch.no_grad():
        for _ in range(10):
            _ = model_inst(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_t = time.perf_counter()
        for _ in range(50):
            _ = model_inst(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end_t = time.perf_counter()
    avg_inf_ms = ((end_t - start_t) / 50) * 1000
    
    complexity_rows.append({
        'Model': display_name,
        'Total Params (M)': f"{total_params / 1e6:.2f}M",
        'Trainable Params (M)': f"{trainable_params / 1e6:.2f}M",
        'Size on Disk': size_str,
        'Inference Speed (ms)': f"{avg_inf_ms:.2f} ms"
    })
    
if len(complexity_rows) > 0:
    complexity_df = pd.DataFrame(complexity_rows)
    print('\n--- TABLE 6: COMPUTATIONAL COMPLEXITY ANALYSIS ---')
    display(complexity_df)
    os.makedirs('results/tables', exist_ok=True)
    complexity_df.to_csv('results/tables/table_6_computational_complexity.csv', index=False)


## THESIS & PPT FIGURE: Preprocessing & LayerCAM Diagnostic Panel
### 4x4 Visual Comparison Grid across all OCT Classes (`CNV`, `DME`, `DRUSEN`, `NORMAL`)
Compares **Original Scan** vs **Bilateral Filter** vs **CLAHE Enhancement** vs **LayerCAM Biomarker Focus**.

In [ ]:
import os, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from src.preprocessing.filters import bilateral_filter, clahe_filter
from src.datasets.dataset import auto_detect_columns, patient_level_split

print('Generating Thesis & PPT Visual Comparison Grid...')
CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
csv_path = 'kermany_dataset_mapping.csv'

if os.path.exists(csv_path):
    df = auto_detect_columns(pd.read_csv(csv_path))
    is_colab = os.path.exists('/content')
    local_kermany = '/content/Kermany'
    local_oct2017 = '/content/OCT2017'
    if is_colab and (os.path.exists(local_kermany) or os.path.exists(local_oct2017)):
        def force_local_path(path_str):
            p = path_str.replace('\\', '/').replace('//', '/')
            parts = p.split('/')
            for folder in ['train', 'val', 'test']:
                if folder in parts:
                    idx = parts.index(folder)
                    subpath = '/'.join(parts[idx:])
                    for candidate in [
                        os.path.join(local_kermany, subpath),
                        os.path.join(local_kermany, 'OCT2017', subpath),
                        os.path.join(local_kermany, 'OCT2017 ', subpath),
                        os.path.join(local_oct2017, subpath)
                    ]:
                        if os.path.exists(candidate):
                            return candidate
            return path_str
        df['image_path'] = df['image_path'].apply(force_local_path)
        
    _, _, test_df = patient_level_split(df)
    
    fig, axes = plt.subplots(4, 4, figsize=(16, 15))
    rows_labels = ['Original Scan', 'Bilateral Filtered', 'CLAHE Enhanced', 'LayerCAM Focus']
    
    for c_idx, c_name in enumerate(CLASSES):
        match_rows = test_df[(test_df['label'] == c_idx) | (test_df['label'] == c_name) | (test_df['label'].astype(str) == str(c_idx))]
        if len(match_rows) > 0:
            img_p = match_rows.iloc[0]['image_path']
            if os.path.exists(img_p):
                raw_gray = cv2.imread(img_p, cv2.IMREAD_GRAYSCALE)
                raw_gray = cv2.resize(raw_gray, (224, 224))
                bilat_gray = bilateral_filter(raw_gray)
                clahe_gray = clahe_filter(bilat_gray)
                
                # LayerCAM heatmap if available
                cam_path = f'outputs/trustoct/layercam/{c_name.lower()}_cam.png'
                if not os.path.exists(cam_path):
                    cam_path = f'outputs/resnet50/layercam/{c_name.lower()}_cam.png'
                
                if os.path.exists(cam_path):
                    cam_img = cv2.imread(cam_path)
                    cam_img = cv2.cvtColor(cam_img, cv2.COLOR_BGR2RGB)
                else:
                    # Create visualization overlay if file not found
                    heatmap_dummy = cv2.applyColorMap(raw_gray, cv2.COLORMAP_JET)
                    orig_rgb = cv2.cvtColor(raw_gray, cv2.COLOR_GRAY2RGB)
                    cam_img = cv2.addWeighted(orig_rgb, 0.6, heatmap_dummy, 0.4, 0)
                    
                # Plot Row 1: Original
                axes[0, c_idx].imshow(raw_gray, cmap='gray')
                axes[0, c_idx].set_title(f'{c_name}', fontsize=14, fontweight='bold', pad=10)
                
                # Plot Row 2: Bilateral
                axes[1, c_idx].imshow(bilat_gray, cmap='gray')
                
                # Plot Row 3: CLAHE
                axes[2, c_idx].imshow(clahe_gray, cmap='gray')
                
                # Plot Row 4: LayerCAM
                axes[3, c_idx].imshow(cam_img)
                
    for r_idx, r_label in enumerate(rows_labels):
        axes[r_idx, 0].set_ylabel(r_label, fontsize=13, fontweight='bold', labelpad=10)
        
    for ax in axes.flatten():
        ax.set_xticks([])
        ax.set_yticks([])
        
    plt.suptitle('TrustOCT Thesis & PPT Presentation Panel: Signal Filtering & Visual Attribution', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    os.makedirs('results/figures', exist_ok=True)
    plt.savefig('results/figures/figure_1_preprocessing_and_layercam_panel.png', dpi=300, bbox_inches='tight')
    print('Saved Thesis & PPT figure to results/figures/figure_1_preprocessing_and_layercam_panel.png')
    plt.show()
else:
    print('Dataset mapping CSV missing. Run dataset setup cell first.')


## TABLE 5: Cross-Dataset External Generalization (OCTID Dataset)
### Evaluates all 6 models trained on Kermany directly on the unseen OCTID dataset (Zero Retraining!)
Proves component-wise contribution (MultiScale -> CBAM -> MixStyle -> EDL) to domain shift robustness.

In [ ]:
import os, torch
import pandas as pd
import numpy as np
from src.models.builder import build_model
from src.evaluation.cross_dataset import run_external_validation

# 1. Download OCTID dataset if not local
octid_root = '/content/OCTID'
if not os.path.exists(octid_root):
    try:
        print('Downloading OCTID dataset from Kaggle...')
        !kaggle datasets download -d gnanapravallika/octid-dataset --unzip -p /content/OCTID 2>/dev/null || true
    except Exception as e:
        print(f'Kaggle download skipped: {e}')

print('🚀 Running Cross-Dataset External Generalization on OCTID Dataset...')

# 2. Build OCTID dataset DataFrame mapping
records_octid = []
class_map_octid = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'amd': 2}

if os.path.exists(octid_root):
    for root, dirs, files in os.walk(octid_root):
        for f in files:
            if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                parent_dir = os.path.basename(root).lower()
                lbl = -1
                for k, v in class_map_octid.items():
                    if k in parent_dir:
                        lbl = v
                        break
                if lbl != -1:
                    records_octid.append({'image_path': os.path.join(root, f), 'label': lbl})
                    
df_octid = pd.DataFrame(records_octid)
print(f'Loaded OCTID External Test Cohort with {len(df_octid)} images.')

# 3. Evaluate all 6 models on OCTID (Zero retraining!)
models_to_test_octid = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'ResNet-50 Baseline'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_resnet50/weights_best.pth', 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_resnet50/weights_best.pth', 'msf_cbam_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_mixstyle_resnet50/weights_best.pth', 'msf_cbam_mixstyle_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_edl_resnet50/weights_best.pth', 'msf_cbam_edl_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/trustoct/weights_best.pth', 'TrustOCT (Proposed)')
]

octid_rows = []

for config_item, weights_path, display_name in models_to_test_octid:
    if os.path.exists(weights_path) and len(df_octid) > 0:
        print(f'Evaluating {display_name} on OCTID...')
        model_inst = build_model(config_item).to(device)
        model_inst.load_state_dict(torch.load(weights_path, map_location=device))
        model_inst.eval()
        
        is_ev = (config_item['model']['head'] == 'evidential')
        results = run_external_validation(
            model=model_inst,
            df_external=df_octid,
            batch_size=32,
            is_evidential=is_ev,
            device_name=str(device)
        )
        
        m = results['metrics']
        octid_rows.append({
            'Model Configuration': display_name,
            'OCTID Accuracy (%)': f"{m['accuracy']*100:.2f}%",
            'Macro F1': f"{m['f1_score']:.4f}",
            'Specificity': f"{m['specificity']:.4f}",
            'ECE (Calibration)': f"{m['ece']:.4f}",
            'Brier Score': f"{m['brier_score']:.4f}"
        })
    else:
        print(f'Skipping {display_name}: weights file {weights_path} or OCTID dataset missing.')

if len(octid_rows) > 0:
    table_5_df = pd.DataFrame(octid_rows)
    print('\n--- TABLE 5: CROSS-DATASET EXTERNAL GENERALIZATION (OCTID DATASET) ---')
    display(table_5_df)
    os.makedirs('results/tables', exist_ok=True)
    table_5_df.to_csv('results/tables/table_5_octid_generalization.csv', index=False)


## Zip All Results for Download

In [ ]:
import shutil

zip_path = '/content/final_paper_results'
if os.path.exists('outputs'):
    shutil.make_archive(zip_path, 'zip', 'outputs')
    print("Zip archive 'final_paper_results.zip' created successfully!")

# Also save to Drive
if os.path.exists(f'{zip_path}.zip'):
    drive_zip = '/content/drive/MyDrive/TrustOCT_Results/final_paper_results.zip'
    shutil.copy(f'{zip_path}.zip', drive_zip)
    print(f'Saved zip to Google Drive: {drive_zip}')